# Knowledge Graph Extraction — Pipeline 3 Passes

**Stratégie de prompt engineering :**
- **Pass 1** : Extraction des types d'entités uniquement (labels)
- **Pass 2** : Extraction des entités avec leurs champs complets
- **Pass 3** : Extraction des relations entre entités connues

**Chunking** : par `=== DOCUMENT X ===` — chaque document est traité indépendamment, puis les résultats sont mergés.

---

## Bloc 1 — Imports et configuration

In [25]:
import os
import json
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

print("Imports OK")
print(f"NVIDIA_API_KEY : {'✓' if os.getenv('NVIDIA_API_KEY') else '✗ MANQUANTE'}")
print(f"NEO4J_URI      : {os.getenv('NEO4J_URI', 'Non défini')}")

Imports OK
NVIDIA_API_KEY : ✓
NEO4J_URI      : bolt://localhost:7687


## Bloc 2 — Initialisation du LLM

In [26]:
llm = ChatNVIDIA(
    model="mistralai/mistral-large-3-675b-instruct-2512",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0,
    max_completion_tokens=32000,
)

## Bloc 3 — Schéma du graphe

In [27]:
SCHEMA = """
NODES (entity: fields):
  Company         : name, activity, certifications
  Supplier        : name
  Agreement       : name, description, type, startdate, enddate
  Document        : reference, title, version, date, owner
  Clients         : name, sector
  Clause          : name, description
  governance_body : name, acronyme, role
  Claim           : claim_type, value, unit, scope, source_doc, source_doc_id, quote
  Roles           : title
  Asset           : type, description, classification
  Algorithm       : name, use
  Protocol        : name, use, source
  Risk            : description, level
  Framework       : name, type, use
  Control         : name, requirement, source, Cycle
  Technology      : name, use, source

RELATIONSHIPS (source -[REL]-> target):
  Agreement       -[for]->              Supplier
  Company         -[Use]->              Agreement
  Supplier        -[HAS]->              Company
  Company         -[HAS]->              Document
  Company         -[HAS]->              Clients
  Company         -[HAS]->              governance_body
  Company         -[has_requirement]->  Clause
  Supplier        -[apply_to]->         Clause
  governance_body -[supervise]->        Roles
  governance_body -[approves]->         Claim
  Claim           -[Concerns]->         Asset
  Claim           -[Asserted_by]->      Roles
  Roles           -[operates]->         Technology
  Roles           -[EXECUTES]->         Control
  Algorithm       -[protects]->         Asset
  Protocol        -[used_by]->          Algorithm
  Protocol        -[mitigates]->        Risk
  Risk            -[targeting]->        Asset
  Framework       -[required_by]->      Control
  Control         -[IMPLEMENTED_BY]->   Technology
  Technology      -[Has_access_to]->    Asset
"""

NODE_LABELS = [
    "Company", "Supplier", "Agreement", "Document", "Clients",
    "Clause", "governance_body", "Claim", "Roles", "Asset",
    "Algorithm", "Protocol", "Risk", "Framework", "Control", "Technology"
]

print(f"Schéma défini : {len(NODE_LABELS)} types de nœuds, 21 types de relations")

Schéma défini : 16 types de nœuds, 21 types de relations


## Bloc 4 — Prompts des 3 passes

**Stratégie de prompt engineering :**
- Pass 1 = tâche simple → liste les labels présents dans le chunk
- Pass 2 = tâche ciblée → extrait uniquement les labels identifiés en pass 1 avec tous leurs champs
- Pass 3 = tâche relationnelle → connaissant les entités, extrait les relations entre elles

In [28]:
# ─────────────────────────────────────────────────────────────────
# PASS 1 — Quels types d'entités sont présents dans ce chunk ?
# ─────────────────────────────────────────────────────────────────
PROMPT_PASS1 = f"""
You are a knowledge-graph analyst. Your task is to SCAN a document chunk
and identify which entity types from the schema are mentioned.

Schema node labels:
{', '.join(NODE_LABELS)}

Rules:
- Return ONLY a JSON array of label strings that are present in the text.
- Include a label only if at least one instance is clearly identifiable.
- No explanation, no markdown, no extra text.

Output format example:
["Company", "Document", "Roles", "Control"]
"""

# ─────────────────────────────────────────────────────────────────
# PASS 2 — Extraire les entités avec leurs champs (labels ciblés)
# ─────────────────────────────────────────────────────────────────
def build_prompt_pass2(detected_labels: list[str]) -> str:
    # Filtre le schéma pour ne montrer que les types détectés → prompt plus court et plus précis
    schema_lines = []
    for line in SCHEMA.strip().split("\n"):
        for label in detected_labels:
            if line.strip().startswith(label):
                schema_lines.append(line)
                break
    focused_schema = "\n".join(schema_lines)

    return f"""
You are a knowledge-graph extraction assistant.
Extract ALL entities of the following types from the document chunk.
Be exhaustive — extract every instance you can find.

Target entity types and their fields:
{focused_schema}

Rules:
- Use ONLY the node labels listed above.
- Extract every field you can find; omit fields that are not mentioned.
- String values only, no nested objects.
- Return ONLY a valid JSON array, no markdown, no explanation.

Output format:
[
  {{"label": "<NodeLabel>", "properties": {{"field": "value"}}}},
  ...
]
"""

# ─────────────────────────────────────────────────────────────────
# PASS 3 — Extraire les relations entre entités connues
# ─────────────────────────────────────────────────────────────────
def build_prompt_pass3(entities: list[dict]) -> str:
    # Résumé compact des entités déjà extraites → ancrage pour la relation
    entity_summary = json.dumps(
        [{"label": e["label"], "key": list(e["properties"].items())[:1]} for e in entities],
        ensure_ascii=False
    )
    return f"""
You are a knowledge-graph relation extractor.
Given the document chunk and the list of already-extracted entities below,
identify all relationships between them that match the schema.

Schema relationships:
{SCHEMA.split('RELATIONSHIPS')[1].strip()}

Already extracted entities (label + identifying key):
{entity_summary}

Rules:
- Use ONLY relationship types from the schema.
- from_key / to_key must match an entity from the list above.
- Return ONLY a valid JSON array, no markdown, no explanation.

Output format:
[
  {{
    "from_label": "<NodeLabel>",
    "from_key":   {{"field": "value"}},
    "rel_type":   "<REL_TYPE>",
    "to_label":   "<NodeLabel>",
    "to_key":     {{"field": "value"}}
  }},
  ...
]
"""

print("Prompts des 3 passes définis")

Prompts des 3 passes définis


## Bloc 5 — Fonctions utilitaires

In [29]:
def call_llm(system_prompt: str, user_text: str) -> str:
    """Appelle le LLM et retourne le contenu brut."""
    messages = [
        ("system", system_prompt),
        ("human", f"Document chunk:\n\n{user_text}"),
    ]
    raw = llm.invoke(messages).content.strip()
    # Nettoyage des balises markdown
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    return raw


def parse_json(raw: str, context: str = ""):
    """Parse JSON avec gestion d'erreur explicite."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"      [JSON ERROR] {context} : {e}")
        return None


def split_by_document(text: str, marker: str = "=== DOCUMENT") -> list[dict]:
    chunks = []
    parts = text.split(marker)
    for part in parts[1:]:
        first_line = part.split("\n")[0].strip().rstrip("===").strip()
        doc_id = f"DOCUMENT {first_line}"
        content = (marker + part).strip()
        
        # Ignorer les chunks trop courts (= table des matières)
        if len(content) < 200:
            continue
            
        chunks.append({"id": doc_id, "text": content})
    return chunks if chunks else [{"id": "FULL", "text": text}]


def merge_results(results: list[dict]) -> dict:
    merged_entities = []
    merged_relations = []
    seen_entities = set()
    seen_relations = set()

    for result in results:
        for entity in result.get("entities", []):
            key = (entity["label"], str(entity.get("properties", {})))
            if key not in seen_entities:
                seen_entities.add(key)
                merged_entities.append(entity)

        for rel in result.get("relationships", []):
            # Ignorer les relations incomplètes
            if not rel.get("from_key") or not rel.get("to_key") or not rel.get("rel_type"):
                continue
            key = (
                rel.get("from_label"),
                str(rel.get("from_key")),
                rel.get("rel_type"),
                rel.get("to_label"),
                str(rel.get("to_key")),
            )
            if key not in seen_relations:
                seen_relations.add(key)
                merged_relations.append(rel)

    return {"entities": merged_entities, "relationships": merged_relations}

## Bloc 6 — Pipeline 3 passes pour un chunk

In [30]:
def extract_chunk_3pass(chunk: dict, all_entities_context: list = None, verbose: bool = True) -> dict:
    """
    Pass 1 : Détection des labels
    Pass 2 : Extraction des entités avec champs
    Pass 3 : Extraction des relations
             - Si all_entities_context est fourni → utilise toutes les entités (version globale)
             - Sinon → utilise uniquement les entités du chunk (version locale)
    """
    doc_id = chunk["id"]
    text   = chunk["text"]

    if verbose:
        print(f"\n   [{doc_id}] ({len(text):,} chars)")

    # ── PASS 1 ────────────────────────────────────────────────────
    if verbose:
        print(f"      Pass 1 — Détection des labels...", end=" ", flush=True)
    raw1 = call_llm(PROMPT_PASS1, text)
    detected_labels = parse_json(raw1, f"{doc_id} Pass1")
    if not detected_labels or not isinstance(detected_labels, list):
        if verbose: print("  Aucun label détecté")
        return {"entities": [], "relationships": []}
    detected_labels = [l for l in detected_labels if l in NODE_LABELS]
    if verbose:
        print(f"{len(detected_labels)} labels : {detected_labels}")

    # ── PASS 2 ────────────────────────────────────────────────────
    if verbose:
        print(f"      Pass 2 — Extraction des entités...", end=" ", flush=True)
    prompt2 = build_prompt_pass2(detected_labels)
    raw2 = call_llm(prompt2, text)
    entities = parse_json(raw2, f"{doc_id} Pass2")
    if not entities or not isinstance(entities, list):
        if verbose: print("  Aucune entité extraite")
        return {"entities": [], "relationships": []}
    if verbose:
        print(f"{len(entities)} entités")

    # ── PASS 3 ────────────────────────────────────────────────────
    if verbose:
        print(f"      Pass 3 — Extraction des relations...", end=" ", flush=True)

    # Contexte = toutes les entités si fourni, sinon uniquement celles du chunk
    context_entities = all_entities_context if all_entities_context is not None else entities

    prompt3 = build_prompt_pass3(context_entities)
    raw3 = call_llm(prompt3, text)
    relationships = parse_json(raw3, f"{doc_id} Pass3")
    if not relationships or not isinstance(relationships, list):
        if verbose: print("  Aucune relation extraite")
        relationships = []
    else:
        if verbose:
            print(f"{len(relationships)} relations")

    return {"entities": entities, "relationships": relationships}

## Bloc 7 — Extraction complète par fichier

In [31]:
def extract_file_3pass(filepath: str, global_relations: bool = False) -> dict:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    chunks = split_by_document(text)
    print(f"   {len(chunks)} chunks trouvés")

    if not global_relations:
        # ── VERSION A : pass 3 chunk par chunk ───────────────────
        print("   Mode : relations locales (par chunk)")
        all_results = []
        for chunk in chunks:
            try:
                result = extract_chunk_3pass(chunk, all_entities_context=None)
                all_results.append(result)
            except Exception as e:
                print(f"        Erreur sur {chunk['id']} : {e}")
                all_results.append({"entities": [], "relationships": []})
        return merge_results(all_results)

    else:
        # ── VERSION B : pass 3 globale après merge des entités ───
        print("   Mode : relations globales (après merge)")

        # Étape 1 — Pass 1 + Pass 2 sur tous les chunks
        all_results = []
        for chunk in chunks:
            try:
                result = extract_chunk_3pass(chunk, all_entities_context=None)
                all_results.append(result)
            except Exception as e:
                print(f"        Erreur sur {chunk['id']} : {e}")
                all_results.append({"entities": [], "relationships": []})

        # Étape 2 — Merge de toutes les entités
        merged_entities = merge_results(all_results)["entities"]
        print(f"\n   Toutes entités mergées : {len(merged_entities)}")

        # Étape 3 — Pass 3 par chunk mais avec TOUTES les entités comme contexte
        print("   Pass 3 globale sur chaque chunk...")
        relation_results = []
        for chunk in chunks:
            try:
                result = extract_chunk_3pass(chunk, all_entities_context=merged_entities)
                relation_results.append({"entities": [], "relationships": result["relationships"]})
            except Exception as e:
                print(f"        Erreur sur {chunk['id']} : {e}")

        # Merge final entités + relations
        final = merge_results(relation_results)
        final["entities"] = merged_entities
        print(f"\n   Total : {len(final['entities'])} entités, {len(final['relationships'])} relations")
        return final

## Bloc 8 — Lancement de l'extraction

In [33]:
DOCUMENTS = [
    "/home/maroua/Desktop/cassiope/SecuraOps High.txt",
    # "/home/maroua/Desktop/cassiope/SafeLink Contradictory.txt",
]

# Version A — relations locales
print("VERSION A — relations locales")
result_A = extract_file_3pass(DOCUMENTS[0], global_relations=False)

# Version B — relations globales
print("\nVERSION B — relations globales")
result_B = extract_file_3pass(DOCUMENTS[0], global_relations=True)

# Comparaison rapide
# Comparaison rapide — avec gestion des champs manquants
def rel_to_key(r):
    return (
        r.get("from_label", ""),
        str(r.get("from_key", {})),
        r.get("rel_type", ""),
        r.get("to_label", ""),
        str(r.get("to_key", {})),
    )

rels_A = set(rel_to_key(r) for r in result_A["relationships"])
rels_B = set(rel_to_key(r) for r in result_B["relationships"])

print(f"\nRelations A : {len(rels_A)}")
print(f"Relations B : {len(rels_B)}")
print(f"Dans B mais pas dans A (cross-doc) : {len(rels_B - rels_A)}")
print(f"Dans A mais pas dans B             : {len(rels_A - rels_B)}")



VERSION A — relations locales
   22 chunks trouvés
   Mode : relations locales (par chunk)

   [DOCUMENT 1 : Politique de Sécurité de l’Information] (9,640 chars)
      Pass 1 — Détection des labels... 11 labels : ['Company', 'Document', 'Clients', 'Clause', 'governance_body', 'Roles', 'Asset', 'Risk', 'Framework', 'Control', 'Technology']
      Pass 2 — Extraction des entités...         Erreur sur DOCUMENT 1 : Politique de Sécurité de l’Information : [504] Gateway Timeout
{'_content': b'', '_content_consumed': True, '_next': None, 'status_code': 504, 'headers': {'Date': 'Thu, 26 Mar 2026 08:13:31 GMT', 'Content-Length': '0', 'Connection': 'keep-alive', 'Access-Control-Expose-Headers': 'nvcf-reqid', 'Nvcf-Reqid': 'b5516e77-f6fa-4aeb-9e4a-a6b32aa672a8', 'Nvcf-Status': 'errored', 'Vary': 'Origin'}, 'raw': <urllib3.response.HTTPResponse object at 0x7405fb7b3ac0>, 'url': 'https://integrate.api.nvidia.com/v1/chat/completions', 'encoding': None, 'history': [], 'reason': 'Gateway Timeout', 'c

KeyboardInterrupt: 

## Bloc 9 — Visualisation des résultats

In [43]:
from collections import Counter

print("\n=== RÉSULTATS VERSION A (PARTIELS) ===")

entities = result_A.get("entities", [])
relationships = result_A.get("relationships", [])

print(f"Entités totales   : {len(entities)}")
print(f"Relations totales : {len(relationships)}")

node_counts = Counter(e['label'] for e in entities)

print("\nTypes d'entités :")
for label, count in sorted(node_counts.items()):
    print(f"   {label:<20} : {count}")


=== RÉSULTATS VERSION A (PARTIELS) ===
Entités totales   : 184
Relations totales : 171

Types d'entités :
   Agreement            : 3
   Asset                : 18
   Clients              : 1
   Company              : 4
   Control              : 50
   Document             : 9
   Framework            : 9
   Protocol             : 5
   Risk                 : 4
   Roles                : 26
   Technology           : 52
   governance_body      : 3


## Bloc 10 — Sauvegarde JSON

In [44]:
import os
import json

# Supposons que result_A contient les résultats de la Version A
nom = os.path.splitext(os.path.basename(DOCUMENTS[0]))[0]
output_file = f"{nom}_graph_3pass_versionA.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result_A, f, indent=2, ensure_ascii=False)

print(f"Sauvegardé : {os.path.abspath(output_file)}")

Sauvegardé : /home/maroua/Desktop/cassiope/SecuraOps High_graph_3pass_versionA.json


## Bloc 11 — Génération des requêtes Cypher

In [45]:
# Fonction utilitaire pour formater les propriétés
def props_str(props: dict) -> str:
    parts = [f"{k}: {json.dumps(v)}" for k, v in props.items()]
    return "{" + ", ".join(parts) + "}"

# Conversion JSON -> Cypher
def json_to_cypher(extracted: dict, company_label: str = None) -> list:
    statements = []
    # Création des noeuds
    for entity in extracted.get("entities", []):
        label = entity["label"]
        props = entity.get("properties", {})
        if props:
            if company_label:
                statements.append(f"MERGE (n:{label}:{company_label} {props_str(props)});")
            else:
                statements.append(f"MERGE (n:{label} {props_str(props)});")
    # Création des relations
    for rel in extracted.get("relationships", []):
        fl = rel["from_label"]
        fk = props_str(rel.get("from_key", {}))
        rt = rel["rel_type"]
        tl = rel["to_label"]
        tk = props_str(rel.get("to_key", {}))
        if company_label:
            statements.append(
                f"MATCH (a:{fl}:{company_label} {fk}), (b:{tl}:{company_label} {tk}) MERGE (a)-[:{rt}]->(b);"
            )
        else:
            statements.append(
                f"MATCH (a:{fl} {fk}), (b:{tl} {tk}) MERGE (a)-[:{rt}]->(b);"
            )
    return statements

# Définir le label de l'entreprise pour Version A
company_label_A = "SecuraOps"
nom_A = os.path.splitext(os.path.basename(DOCUMENTS[0]))[0]

# Générer les statements Cypher uniquement pour Version A
cypher_A = json_to_cypher(result_A, company_label=company_label_A)
print(f"{nom_A} : {len(cypher_A)} statements Cypher Version A")

SecuraOps High : 355 statements Cypher Version A


## Bloc 12 — Envoi vers Neo4j

In [47]:
import os
import json
from neo4j import GraphDatabase

#-------------------------
# Fonction utilitaire pour formater les propriétés
#-------------------------
def props_str(props: dict) -> str:
    if not props:
        return ""
    parts = [f"{k}: {json.dumps(v)}" for k, v in props.items()]
    return "{" + ", ".join(parts) + "}"

#-------------------------
# Conversion JSON -> Cypher
#-------------------------
def json_to_cypher(extracted: dict, company_label: str = None) -> list:
    statements = []

    # Création des noeuds
    for entity in extracted.get("entities", []):
        label = entity["label"]
        props = entity.get("properties", {})
        prop_str = props_str(props)
        if company_label:
            if prop_str:
                statements.append(f"MERGE (n:{label}:{company_label} {prop_str});")
            else:
                statements.append(f"MERGE (n:{label}:{company_label});")
        else:
            if prop_str:
                statements.append(f"MERGE (n:{label} {prop_str});")
            else:
                statements.append(f"MERGE (n:{label});")

    # Création des relations
    for rel in extracted.get("relationships", []):
        fl = rel["from_label"]
        fk = props_str(rel.get("from_key", {}))
        rt = rel["rel_type"]
        tl = rel["to_label"]
        tk = props_str(rel.get("to_key", {}))
        if company_label:
            match_a = f"(a:{fl}:{company_label} {fk})" if fk else f"(a:{fl}:{company_label})"
            match_b = f"(b:{tl}:{company_label} {tk})" if tk else f"(b:{tl}:{company_label})"
        else:
            match_a = f"(a:{fl} {fk})" if fk else f"(a:{fl})"
            match_b = f"(b:{tl} {tk})" if tk else f"(b:{tl})"
        statements.append(f"MATCH {match_a}, {match_b} MERGE (a)-[:{rt}]->(b);")

    return statements

#-------------------------
# Définir le label de l'entreprise et le fichier
#-------------------------
company_label_A = "SecuraOps"
nom_A = "VersionA"  # tu peux mettre le nom du fichier ici
# result_A doit être ton JSON extrait
# Ex: result_A = json.load(open("result_A.json"))

# Générer les statements Cypher
cypher_A = json_to_cypher(result_A, company_label=company_label_A)
cypher_by_file = {nom_A: cypher_A}

print(f"{nom_A} : {len(cypher_A)} statements Cypher Version A")

#-------------------------
# Connexion à Neo4j
#-------------------------
uri      = os.getenv("NEO4J_URI", "bolt://localhost:7687")
user     = os.getenv("NEO4J_USER", "neo4j")
password = os.getenv("NEO4J_PASSWORD", "")

driver = GraphDatabase.driver(uri, auth=(user, password))

#-------------------------
# Vider la base
#-------------------------
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("Base vidée")

#-------------------------
# Envoi des statements
#-------------------------
for nom, stmts in cypher_by_file.items():
    print(f"\nEnvoi de {nom} vers Neo4j...")
    errors = 0
    with driver.session() as session:
        for stmt in stmts:
            try:
                print("Executing:", stmt)
                session.run(stmt)
            except Exception as e:
                print("Erreur:", e)
                errors += 1
    print(f"   {len(stmts) - errors} statements OK")
    if errors:
        print(f"   {errors} erreurs")

driver.close()
print("\nTerminé !")

VersionA : 355 statements Cypher Version A
Base vidée

Envoi de VersionA vers Neo4j...
Executing: MERGE (n:Document:SecuraOps {reference: "DATA-PO-001", title: "Politique de Classification des Donn\u00e9es", version: "1.6", date: "Octobre 2025", owner: "D\u00e9l\u00e9gu\u00e9 \u00e0 la Protection des Donn\u00e9es (DPO)"});
Executing: MERGE (n:Company:SecuraOps {name: "SecuraOps"});
Executing: MERGE (n:Technology:SecuraOps {name: "chiffrement fort", use: "transferts de donn\u00e9es Restreintes"});
Executing: MERGE (n:Technology:SecuraOps {name: "effacement s\u00e9curis\u00e9 (3 passes DoD)", use: "destruction des donn\u00e9es"});
Executing: MERGE (n:Technology:SecuraOps {name: "broyage physique", use: "destruction des donn\u00e9es"});
Executing: MERGE (n:Framework:SecuraOps {name: "politique de r\u00e9tention", type: "Conservation", use: "d\u00e9finir les dur\u00e9es de conservation"});
Executing: MERGE (n:Control:SecuraOps {name: "marquage des documents", requirement: "Chaque document,